In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests

# -----------------------------
# Config
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
vocab_size = 256          # byte-level
d_model = 256
n_heads = 4
n_layers = 4
block_size = 256 # Increased context window
ff_hidden = 1024           # dense FFN hidden size (4 * d_model)

batch_size = 32
lr = 3e-4
num_steps = 200000 # Increased training steps
print_every = 1000

In [3]:
# -----------------------------
# Download Gutenberg Shakespeare
# -----------------------------
url = "https://www.gutenberg.org/files/100/100-0.txt"
print("Downloading Shakespeare from Gutenberg...")
response = requests.get(url)
response.raise_for_status()
corpus = response.text

# Convert to bytes → tensor
data_bytes = corpus.encode("utf-8")
data = torch.tensor(list(data_bytes), dtype=torch.long)

print(f"Dataset loaded: {len(data)} bytes")

Dataset loaded: 5422721 bytes


In [4]:
# -----------------------------
# Byte-level batch sampler
# -----------------------------
def get_batch():
    # The 'high' parameter to randint is exclusive. We want to sample indices `i`
    # such that `i + block_size + 1` (for target `y`) does not exceed `len(data)`.
    # So, the maximum `i` should be `len(data) - block_size - 1`.
    # Therefore, `high` must be `len(data) - block_size`.
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

In [5]:
# -----------------------------
# Model components
# -----------------------------
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = F.softmax(att, dim=-1)
        y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(y)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_model)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_hidden):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, ff_hidden)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class TinyDenseLLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers,
                 block_size, ff_hidden):
        super().__init__()
        self.block_size = block_size

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, ff_hidden)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)

        x = self.token_emb(idx) + self.pos_emb(pos)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(B * T, -1),
                targets.view(B * T)
            )
        return logits, loss

In [6]:
# -----------------------------
# Instantiate model
# -----------------------------
# Clear GPU cache before instantiation to prevent lingering memory issues
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = TinyDenseLLM(
    vocab_size=vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    block_size=block_size,
    ff_hidden=ff_hidden,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

In [7]:
# -----------------------------
# Generation
# -----------------------------
model.eval()

def generate(model, start_text, max_new_tokens=200):
    with torch.no_grad():
        context = torch.tensor(list(start_text.encode("utf-8")), dtype=torch.long)[None, :].to(device)
        for _ in range(max_new_tokens):
            # Crop context to block_size if it's longer for positional embeddings
            idx_cond = context if context.shape[1] <= model.block_size else context[:, -model.block_size:]

            # Get predictions
            logits, _ = model(idx_cond)

            # Focus on the logits for the last timestep
            logits = logits[:, -1, :]

            # Sample the next token
            next_token = torch.distributions.Categorical(logits=logits).sample()

            # Append sampled token to the running sequence
            context = torch.cat([context, next_token.unsqueeze(0)], dim=1)

            # Break if max_new_tokens reached
            if context.shape[1] >= max_new_tokens + len(start_text):
                break
        return bytes(context[0].tolist()).decode("utf-8", errors="ignore")

In [ ]:
# -----------------------------
# Training loop
# -----------------------------
model.train()
for step in range(1, num_steps + 1):
    x, y = get_batch()

    logits, loss = model(x, y)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % print_every == 0:
        print(f"step {step:4d} | loss {loss.item():.4f}")
        seed = "To be, or not to be"
        print("\n=== SAMPLE ===")
        print(generate(model, seed), "\n")

step 1000 | loss 0.0138

=== SAMPLE ===
To be, or not to be )orQor, H)r roke,? :roo2Trr F oroorr?Qor?-jo-rrr?UKirt?
r?2Z"rrrrrr, Hr?rr?9rrrrr?
rorrr?rrrrorrrr?rrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr?
rrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr 

step 2000 | loss 0.0108

=== SAMPLE ===
To be, or not to be 1? r? r 1?e  _ ? ? _5 9-  5?? 25? 25? ? ? 4 5? 7? ? ?  Ht?; ? ?? ? ?  ???? r??? ?r? ? ? 1? ? ?? j4 ? ? ? ?  ? ?  ? ? 5?  ?  ? ?a? 2? ? 6? ?  ?  ?  c? ? ? ? ??  ? ?? 2  ? ?? ? 4? ?
? 4?? ) ?? K? ? 

step 3000 | loss 0.0109

=== SAMPLE ===
To be, or not to be t or oontt? Ht)ottN4 N1t o, D t 3oNoo)-tDDN Dto6NNDNDNN6tt325952116JN5)oNNNEDNDYNDDNSDo5DNDNDDNDNNZNDNDNNDDNDNDNDNNDNDDDNNEDDDDDNDDCDDDNNNDNNDDDNNNDDDDNNDNDDDDDYDDDDDNDDMNDDNDDNNDDDDNDDDDDNDNNDD 

step 4000 | loss 0.0092

=== SAMPLE ===
To be, or not to be  tortoo, To   oT ,  o  f ow  toto  oozo 4ozoo o ootoo ooto4o  o  oooo o ooo Qo ooooooooooooooooo oo  ooooo oo ooooooHooooooooo oooo onoooooooo oooo oooooooooooooooooooo

In [ ]:
# -----------------------------
# Generation
# -----------------------------
model.eval()

def generate(model, start_text, max_new_tokens=200):
    with torch.no_grad():
        context = torch.tensor(list(start_text.encode("utf-8")), dtype=torch.long)[None, :].to(device)
        while context.shape[1] < max_new_tokens and context.shape[1] < block_size:
            logits, _ = model(context)
            next_token = torch.distributions.Categorical(logits=logits[:, -1, :]).sample()
            context = torch.cat([context, next_token.unsqueeze(0)], dim=1)
        return bytes(context[0].tolist()).decode("utf-8", errors="ignore")

seed = "To be, or not to be"
print("\n=== SAMPLE ===")
print(generate(model, seed))